# Sequence Labelling

In this session we will build an HMM model for PoS-tagging and then CRF and neural models for Named Entity Recognition.

## Building a simple Hidden Markov Model

In this first part of the lab we will build a very simple bigram HMM using probability estimates over the Brown corpus, which is Part-of-Speech tagged.

Recall from course 6: probability estimates (with MLE estimation) can be calculated by dividing the number of occurrences of a bigram by the number of occurrences of the first word.

First of all, we import the corpus where we will estimate the probabilities:


In [33]:
import nltk
from nltk.corpus import brown
nltk.download('brown')

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\Alexander\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

This corpus is in the form of sequences of sentences, where each sentence is made by a sequence of pairs (word, POS-tag), like this:

In [34]:
brown.tagged_sents()

[[('The', 'AT'), ('Fulton', 'NP-TL'), ('County', 'NN-TL'), ('Grand', 'JJ-TL'), ('Jury', 'NN-TL'), ('said', 'VBD'), ('Friday', 'NR'), ('an', 'AT'), ('investigation', 'NN'), ('of', 'IN'), ("Atlanta's", 'NP$'), ('recent', 'JJ'), ('primary', 'NN'), ('election', 'NN'), ('produced', 'VBD'), ('``', '``'), ('no', 'AT'), ('evidence', 'NN'), ("''", "''"), ('that', 'CS'), ('any', 'DTI'), ('irregularities', 'NNS'), ('took', 'VBD'), ('place', 'NN'), ('.', '.')], [('The', 'AT'), ('jury', 'NN'), ('further', 'RBR'), ('said', 'VBD'), ('in', 'IN'), ('term-end', 'NN'), ('presentments', 'NNS'), ('that', 'CS'), ('the', 'AT'), ('City', 'NN-TL'), ('Executive', 'JJ-TL'), ('Committee', 'NN-TL'), (',', ','), ('which', 'WDT'), ('had', 'HVD'), ('over-all', 'JJ'), ('charge', 'NN'), ('of', 'IN'), ('the', 'AT'), ('election', 'NN'), (',', ','), ('``', '``'), ('deserves', 'VBZ'), ('the', 'AT'), ('praise', 'NN'), ('and', 'CC'), ('thanks', 'NNS'), ('of', 'IN'), ('the', 'AT'), ('City', 'NN-TL'), ('of', 'IN-TL'), ('Atlant

We recall (from the course) that a Hidden Markov Model is composed by:

- A set of $N$ states $Q = \{q_1, q_2, \ldots, q_N\}$
- A transition probability matrix $A=a_{11}\ldots a_{ij} \ldots a_{NN}$, where each $a_{ij}$ represents the probability of transitioning from state $q_i$ to $q_j$; note that $\sum_{j=1}^N{a_{ij}} = 1 \forall i$
- A sequence of $T$ observations $O = o_1, o_2, \ldots o_T$, each one drawn from a vocabulary of size $V=v_1, v_2, \ldots, v_M$, of size $M$;
- A sequence of *observation likelihoods* $B=b_i(o_t)$, also called **emission probabilities**, each expressing the probability of an observation $o_t$ being generated from a state $q_i$;
- Finally, an initial probability distribution $\Pi = \pi_i, \pi_2, \ldots, \pi_N$ where $\pi_i$ indicates the probability that the Markov chain will start from state $q_i$. Some states $q_j$ may have $\pi_j = 0$, meaning that they cannot be initial states. Also, $\sum_{i=1}^N{\pi_i}=1$.

In our case, the set of states $Q$ is made by the vocabulary of labels (the POS-tags). The vocabulary $V$ corresponds to the word vocabulary (i.e. all the set of different words that appear in our corpus). The observations correspond to the sentences in the corpus.

We will now split our corpus in a training and test set:



In [36]:
corpus = brown.tagged_sents()

training = corpus[:-10]
testing = corpus[-10:]
print(training[0])
print(testing[0])

[('The', 'AT'), ('Fulton', 'NP-TL'), ('County', 'NN-TL'), ('Grand', 'JJ-TL'), ('Jury', 'NN-TL'), ('said', 'VBD'), ('Friday', 'NR'), ('an', 'AT'), ('investigation', 'NN'), ('of', 'IN'), ("Atlanta's", 'NP$'), ('recent', 'JJ'), ('primary', 'NN'), ('election', 'NN'), ('produced', 'VBD'), ('``', '``'), ('no', 'AT'), ('evidence', 'NN'), ("''", "''"), ('that', 'CS'), ('any', 'DTI'), ('irregularities', 'NNS'), ('took', 'VBD'), ('place', 'NN'), ('.', '.')]
[('you', 'PPSS'), ("can't", 'MD*'), ('very', 'QL'), ('well', 'RB'), ('sidle', 'VB'), ('up', 'IN'), ('to', 'IN'), ('people', 'NNS'), ('on', 'IN'), ('the', 'AT'), ('street', 'NN'), ('and', 'CC'), ('ask', 'VB'), ('if', 'CS'), ('they', 'PPSS'), ('want', 'VB'), ('to', 'TO'), ('buy', 'VB'), ('a', 'AT'), ('hot', 'JJ'), ('Bodhisattva', 'NP'), ('.', '.')]


**Exercise 1**: Extract $Q$ and $V$ from the Brown corpus and determine their respective size

In [37]:
#insert your solution here
# Initialisation des ensembles pour garantir l'unicité des éléments
Q_set = set()
V_set = set()

# Itération sur le corpus d'entraînement pour extraire les mots et les tags
for sentence in training:
    for word, tag in sentence:
        Q_set.add(tag)
        V_set.add(word)

# Conversion en listes pour permettre l'indexation lors de la construction des matrices
Q = list(Q_set)
V = list(V_set)

# Calcul de la cardinalité des ensembles
N = len(Q)
M = len(V)

print(f"Taille de l'ensemble des états Q (N) : {N}")
print(f"Taille du vocabulaire V (M) : {M}")

Taille de l'ensemble des états Q (N) : 472
Taille du vocabulaire V (M) : 56043


**Exercise 2**: Create the matrices ($A$, $B$ and $\Pi$) by using the probabilities estimated on the training set; since we are considering bigrams, the probabilities of the transition matrix will be calculated as $\frac{count(t_{-1}, t)}{count(t_{-1})}$.

*Important*: you will need to add smoothing (for instance Lidstone with $k=0.1$) on $B$ otherwise the output will be $0$.

In [38]:
#insert your solution here
import numpy as np

# 1. Création des mappings (utiles pour cet exercice et pour la fonction Viterbi)
tag_to_index = {tag: i for i, tag in enumerate(Q)}
word_to_index = {word: i for i, word in enumerate(V)}

N = len(Q)
M = len(V)

# 2. Initialisation des matrices avec des zéros
pi = np.zeros(N)
A = np.zeros((N, N))
B = np.zeros((N, M))

# 3. Comptage des occurrences sur le corpus d'entraînement
S = len(training)
for sentence in training:
    if not sentence:
        continue
        
    # Comptage pour l'état initial
    first_word, first_tag = sentence[0]
    pi[tag_to_index[first_tag]] += 1
    
    for i in range(len(sentence)):
        word, tag = sentence[i]
        t_idx = tag_to_index[tag]
        w_idx = word_to_index[word]
        
        # Comptage pour les émissions
        B[t_idx, w_idx] += 1
        
        # Comptage pour les transitions
        if i > 0:
            prev_tag = sentence[i-1][1]
            prev_t_idx = tag_to_index[prev_tag]
            A[prev_t_idx, t_idx] += 1

# 4. Normalisation et Lissage
# Probabilités initiales
pi = pi / S

# Probabilités de transition (Division ligne par ligne pour que la somme de chaque ligne vaille 1)
row_sums_A = A.sum(axis=1, keepdims=True)
# Utilisation de np.divide avec l'argument 'where' pour éviter la division par zéro si un état n'apparaît jamais comme état précédent
A = np.divide(A, row_sums_A, out=np.zeros_like(A), where=(row_sums_A != 0))

# Probabilités d'émission avec lissage de Lidstone (k = 0.1)
k = 0.1
B = (B + k) / (B.sum(axis=1, keepdims=True) + k * M)
#Expected output:
#pi matrix such that pi[i] is the probability of starting in state q_i
#A matrix (Q x Q sized) such that A[i][j] is the probability of moving from state q_i to state q_j
#B matrix (Q x O sized) such that B[i][j] is the probability of state q_i to emit the word (observation) o_j


We have now a model and we can estimate its performance over the test set.

To do this, we need the Viterbi algorithm for the decoding. To help you, an implementation of Viterbi is provided:
(note: to use this version you need to assign a numeric id to each word and tag, if you haven't already)

In [42]:
"""
params is a triple (pi, A, B) where
pi = initial probability distribution over states
A = transition probability matrix
B = emission probability matrix

observations is the sequence of observations (in our case, the observed words)

the function returns the optimal sequence of states and its score
"""
import numpy as np


def viterbi(params, observations):
    pi, A, B = params
    M = len(observations)
    S = pi.shape[0]

    alpha = np.zeros((M, S))
    alpha[:,:] = float('-inf') #cases that have not been treated
    backpointers = np.zeros((M, S), 'int')

    # base case
    alpha[0, :] = pi * B[:,observations[0]]

    # recursive case
    for t in range(1, M):
        for s2 in range(S):
            for s1 in range(S):
                score = alpha[t-1, s1] * A[s1, s2] * B[s2, observations[t]]
                if score > alpha[t, s2]:
                    alpha[t, s2] = score
                    backpointers[t, s2] = s1
    # now follow backpointers to resolve the state sequence
    ss = []
    ss.append(np.argmax(alpha[M-1,:]))
    for i in range(M-1, 0, -1):
        ss.append(backpointers[i, ss[-1]])

    return list(reversed(ss)), np.max(alpha[M-1,:])

#Example:

#original sentence: you can't very well sidle up to people on the street and ask if they want to buy a hot Bodhisattva .
#sentence as sequence of word indexes:
word_index=[42350, 44913, 3024, 50638, 15858, 16209, 36949, 31092, 28334, 45518, 22719, 26179, 32651, 52996, 25205, 16840, 36949, 1402, 46003, 10606, 19795, 3739]

predicted, score = viterbi((pi, A, B), word_index)
print(predicted)

#predicted will be a sequence of tag indexes:
#[12, 55, 86, 39, 29, 4, 70, 7, 14, 7, 0, 6, 21, 28, 12, 55, 28, 27, 28, 0, 9, 14, 15]

[152, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272, 272]


In [45]:
# 1. Ajout du token <UNK> au vocabulaire et au mapping
UNK_TOKEN = "<UNK>"
UNK_INDEX = len(V)
word_to_index[UNK_TOKEN] = UNK_INDEX

# 2. Extraction des probabilités d'émission pour les mots de fréquence nulle
unk_column = np.min(B, axis=1, keepdims=True)

# 3. Concaténation de la colonne <UNK> à la matrice B
B = np.hstack((B, unk_column))

# 4. Renormalisation des lignes de B pour garantir que la somme vaille 1
B = B / B.sum(axis=1, keepdims=True)

In [46]:
#Example of results calculation

testing_formated = []
for sentence in testing:
    sent = ""
    words_index = [] #vector of word indices to be passed to Viterbi
    true_label= [] #vector of the true labels from labeled corpus
    for word,tag in sentence:
        idx = word_to_index.get(word, word_to_index["<UNK>"])
        words_index.append(idx) #word_to_index is a dictionary mapping a word to its index
        true_label.append(tag)
        sent=sent+" "+word
    testing_formated.append((words_index,true_label,sent))


for word_index,labels,sentence in testing_formated:
    print("  The sentence is:",sentence)
    print("   ##TRUE##    ##PRED##")
    predicted, score = viterbi((pi, A, B), word_index) #call the viterbi decoder
    for i,true_label in enumerate(labels):
        predicted_label = Q[predicted[i]] #Q here is the vector of tags, so that at Q[i] we have the i_th tag in literal form
        print("      "+true_label+"         "+predicted_label)


  The sentence is:  you can't very well sidle up to people on the street and ask if they want to buy a hot Bodhisattva .
   ##TRUE##    ##PRED##
      PPSS         PPSS
      MD*         MD*
      QL         QL
      RB         RB
      VB         VBD
      IN         RP
      IN         IN
      NNS         NNS
      IN         IN
      AT         AT
      NN         NN
      CC         CC
      VB         VB
      CS         CS
      PPSS         PPSS
      VB         VB
      TO         TO
      VB         VB
      AT         AT
      JJ         JJ
      NP         .
      .         .
  The sentence is:  Additionally , since you're going to be hors de combat pretty soon with sprue , yaws , Delhi boil , the Granville wilt , liver fluke , bilharziasis , and a host of other complications of the hex you've aroused , you mustn't expect to be lionized socially .
   ##TRUE##    ##PRED##
      RB         RB
      ,         ,
      CS         CS
      PPSS+BER         PPSS+BER
      VBG     

**Exercise 3**: calculate Precision, Recall and F-measure for the bigram model

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report

y_true = []
y_pred = []
print(testing_formated[0])

# 1. Agrégation des prédictions et des étiquettes réelles
for word_index, labels, sentence in testing_formated:
    # Décodage par l'algorithme de Viterbi
    predicted, score = viterbi((pi, A, B), word_index)
    
    for i, true_label in enumerate(labels):
        # Conversion de l'index de l'état prédit vers le label textuel
        predicted_label = Q[predicted[i]]
        
        y_true.append(true_label)
        y_pred.append(predicted_label)

# 2. Calcul des métriques globales (Moyenne pondérée par la fréquence des classes)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, 
    y_pred, 
    average='weighted', 
    zero_division=0
)

print("=== Évaluation globale du modèle HMM (Moyenne Pondérée) ===")
print(f"Précision : {precision:.4f}")
print(f"Rappel    : {recall:.4f}")
print(f"F-Mesure  : {f1:.4f}\n")

# 3. Rapport détaillé par tag POS
print("=== Rapport détaillé par classe ===")
print(classification_report(y_true, y_pred, zero_division=0))

([34382, 50428, 49154, 2277, 56043, 13471, 13084, 11749, 53198, 18099, 4211, 4361, 23258, 41272, 41283, 387, 13084, 24329, 2852, 26680, 56043, 10834], ['PPSS', 'MD*', 'QL', 'RB', 'VB', 'IN', 'IN', 'NNS', 'IN', 'AT', 'NN', 'CC', 'VB', 'CS', 'PPSS', 'VB', 'TO', 'VB', 'AT', 'JJ', 'NP', '.'], " you can't very well sidle up to people on the street and ask if they want to buy a hot Bodhisattva .")
=== Évaluation globale du modèle HMM (Moyenne Pondérée) ===
Précision : 0.8612
Rappel    : 0.8703
F-Mesure  : 0.8611

=== Rapport détaillé par classe ===
              precision    recall  f1-score   support

          ''       0.00      0.00      0.00         0
           ,       1.00      1.00      1.00        24
          --       1.00      1.00      1.00         2
           .       0.88      1.00      0.93         7
          AP       1.00      1.00      1.00         4
          AT       0.96      1.00      0.98        26
       AT-HL       0.00      0.00      0.00         1
          BE      

: 

## Using NLTK's HMM implementation

We will compare now our model built from scratch to the implementation provided by NLTK:

In [1]:
import nltk
from nltk.tag import hmm

trainer = hmm.HiddenMarkovModelTrainer(states = Q, symbols = V)

model = trainer.train_supervised(training, estimator=lambda fd, bins: hmm.LidstoneProbDist(fd, 0.1, bins))

for sent in testing:
    u_sent=[]
    for word, tag in sent:
        u_sent.append(word)
    tagged=model.tag(u_sent)
    print(sent)
    print(tagged)


NameError: name 'Q' is not defined

**Exercise 4**: Calculate precision, recall and F-measure and compare them to the results that you obtained with your model. If the results are similar we can expect the NLTK tagger to be bigram-based too.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report

y_true_nltk = []
y_pred_nltk = []

# 1. Génération des prédictions via le modèle NLTK
for sent in testing:
    u_sent = []
    true_labels = []
    
    # Extraction des mots et des vrais tags
    for word, tag in sent:
        u_sent.append(word)
        true_labels.append(tag)
        
    # Prédiction de la séquence par NLTK (retourne une liste de tuples (mot, tag))
    tagged = model.tag(u_sent)
    
    # Séparation et stockage pour l'évaluation
    for i in range(len(true_labels)):
        y_true_nltk.append(true_labels[i])
        y_pred_nltk.append(tagged[i][1])

# 2. Calcul des métriques globales pondérées
precision_nltk, recall_nltk, f1_nltk, _ = precision_recall_fscore_support(
    y_true_nltk, 
    y_pred_nltk, 
    average='weighted', 
    zero_division=0
)

print("=== Évaluation du modèle NLTK (Moyenne Pondérée) ===")
print(f"Précision : {precision_nltk:.4f}")
print(f"Rappel    : {recall_nltk:.4f}")
print(f"F-Mesure  : {f1_nltk:.4f}\n")

# print(classification_report(y_true_nltk, y_pred_nltk, zero_division=0))

=== Évaluation du modèle NLTK (Moyenne Pondérée) ===
Précision : 0.8612
Rappel    : 0.8703
F-Mesure  : 0.8611



## Named Entity Recognition with Conditional Random Fields

For this exercise we will need to use the sklearn_crfsuite package. If it is not installed, it can be installed using pip with ```pip install sklearn-crfsuite```.

We will work on a Kaggle dataset named ```ner_dataset.csv``` (it should be in the same directory as the notebook).

Pandas can be used to read the content of the file:

In [20]:
import pandas as pd

data = pd.read_csv("ner_dataset.csv", encoding="latin1")
data = data.ffill() #repeat sentence number on each row

words = list(set(data["Word"].values)) #vocabulary V
n_words = len(words)

print(words[:10])
print(n_words)

['technology-intensive', '1.38.53', 'tetrahydrogestrinone', 'sanctuaries', 'Pedro', 'Balboa', 'Alpine', 'redeployment', 'guy', 'African-Union']
35177


We provide you with some code that can read the sentences and produce the features in the format required by crf_suite. The ```SentenceGetter``` class transforms sentences into sequences of ```(word, POS, tag)``` triples

In [21]:
class SentenceGetter(object):

    def __init__(self, data):
        self.n_sent = 1
        self.data = data
        self.empty = False
        agg_func = lambda s: [(w, p, t) for w, p, t in zip(s["Word"].values.tolist(),
                                                           s["POS"].values.tolist(),
                                                           s["Tag"].values.tolist())]
        self.grouped = self.data.groupby("Sentence #").apply(agg_func)
        self.sentences = [s for s in self.grouped]

    def get_next(self):
        try:
            s = self.grouped["Sentence: {}".format(self.n_sent)]
            self.n_sent += 1
            return s
        except:
            return None
#load data
getter = SentenceGetter(data) #transform sentences into sequences of (Word, POS, Tag)
sentences = getter.sentences

C:\Users\Alexander\AppData\Local\Temp\ipykernel_29144\985691969.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  self.grouped = self.data.groupby("Sentence #").apply(agg_func)


The next function allows us to define features that are used in the CRF. The features are stored in a dictionary.

In [22]:
def word2features(sent, i):
    """
    input:
       sent: sentence in the format of sequence of (Word, POS, Tag) triples
       i: position in the sentence
    output:
       features: a dictionary mapping the feature name into a value
    """
    word = sent[i][0]
    postag = sent[i][1]

    features = { #features related to the current position
        'bias': 1.0,
        'word.lower()': word.lower(),
        'postag': postag,
    }
    if i > 0: #features related to preceding word/tag
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:postag': postag1,
        })
    else:
        features['BOS'] = True #feature for Beginning of Sentence

    if i < len(sent)-1: #features related to the following word/tag
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:postag': postag1,
        })
    else:
        features['EOS'] = True #feature for end of sentence

    return features


def sent2features(sent):
    #transforms the sentence in a sequence of features
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    #transforms the sentence in a sequence of labels
    return [label for token, postag, label in sent]

def sent2tokens(sent):
    #transforms the sentence in a sequence of tokens (removes POS tags and labels)
    return [token for token, postag, label in sent]


We can now build the features and label vectors, and create a CRF model:

In [ ]:
!pip install sklearn_crfsuite

In [25]:
X = [sent2features(s) for s in sentences]
y = [sent2labels(s) for s in sentences]

from sklearn_crfsuite import CRF
crf = CRF(algorithm='lbfgs',  max_iterations=100)


This will create a model with gradient descent algorithm ("lbfgs") and a limit of $100$ iterations.

Now we build the model and evaluate it on a 66/33 split between training and testing:

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
crf.fit(X_train, y_train)
pred=crf.predict(X_test)

n_pred = [item for sublist in pred for item in sublist]
n_test = [item for sublist in y_test for item in sublist]

report = classification_report(y_pred=n_pred, y_true=n_test)
print(report)


              precision    recall  f1-score   support

       B-art       0.40      0.01      0.03       137
       B-eve       0.69      0.31      0.42       111
       B-geo       0.83      0.91      0.87     12357
       B-gpe       0.98      0.82      0.89      5226
       B-nat       0.60      0.09      0.15        69
       B-org       0.78      0.67      0.72      6762
       B-per       0.83      0.79      0.81      5649
       B-tim       0.93      0.84      0.88      6650
       I-art       0.38      0.02      0.05       124
       I-eve       0.56      0.21      0.31        89
       I-geo       0.80      0.79      0.80      2433
       I-gpe       0.95      0.33      0.49        55
       I-nat       0.33      0.05      0.08        21
       I-org       0.76      0.78      0.77      5545
       I-per       0.85      0.87      0.86      5730
       I-tim       0.81      0.74      0.78      2110
           O       0.99      0.99      0.99    292571

    accuracy              

The report shows accuracy stats for all classes, but we are not interested in the **O** class.

**Exercise 6.**: try to implement additional features to increase scores.

In [27]:
def word2features(sent, i):
    """
    Extrait les caractéristiques locales et contextuelles pour le mot à l'indice i.
    """
    word = sent[i][0]
    postag = sent[i][1]

    # Caractéristiques du mot courant
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],       # Suffixe de taille 3
        'word[-2:]': word[-2:],       # Suffixe de taille 2
        'word.isupper()': word.isupper(), # Indicateur d'acronyme
        'word.istitle()': word.istitle(), # Indicateur de nom propre
        'word.isdigit()': word.isdigit(), # Indicateur numérique
        'postag': postag,
        'postag[:2]': postag[:2],     # Catégorie POS principale
    }
    
    # Caractéristiques du mot précédent (i-1)
    if i > 0:
        word1 = sent[i-1][0]
        postag1 = sent[i-1][1]
        features.update({
            '-1:word.lower()': word1.lower(),
            '-1:word.istitle()': word1.istitle(),
            '-1:word.isupper()': word1.isupper(),
            '-1:postag': postag1,
            '-1:postag[:2]': postag1[:2],
        })
    else:
        features['BOS'] = True # Beginning Of Sentence

    # Caractéristiques du mot suivant (i+1)
    if i < len(sent)-1:
        word1 = sent[i+1][0]
        postag1 = sent[i+1][1]
        features.update({
            '+1:word.lower()': word1.lower(),
            '+1:word.istitle()': word1.istitle(),
            '+1:word.isupper()': word1.isupper(),
            '+1:postag': postag1,
            '+1:postag[:2]': postag1[:2],
        })
    else:
        features['EOS'] = True # End Of Sentence

    return features



In [28]:
X = [sent2features(s) for s in sentences]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
crf.fit(X_train, y_train)
pred=crf.predict(X_test)

n_pred = [item for sublist in pred for item in sublist]
n_test = [item for sublist in y_test for item in sublist]

report = classification_report(y_pred=n_pred, y_true=n_test)
print(report)

              precision    recall  f1-score   support

       B-art       0.31      0.03      0.05       137
       B-eve       0.51      0.33      0.40       111
       B-geo       0.85      0.91      0.88     12357
       B-gpe       0.96      0.93      0.94      5226
       B-nat       0.56      0.26      0.36        69
       B-org       0.80      0.70      0.75      6762
       B-per       0.82      0.81      0.81      5649
       B-tim       0.91      0.87      0.89      6650
       I-art       0.25      0.04      0.07       124
       I-eve       0.38      0.29      0.33        89
       I-geo       0.81      0.81      0.81      2433
       I-gpe       0.93      0.51      0.66        55
       I-nat       0.40      0.10      0.15        21
       I-org       0.79      0.78      0.79      5545
       I-per       0.82      0.91      0.86      5730
       I-tim       0.81      0.76      0.78      2110
           O       0.99      0.99      0.99    292571

    accuracy              

## LSTM model with CRF head

The following code is an implementation of a CRF head.

In the head we need to complete the calculation of the path score. The path score of a given tag sequence in a linear-chain CRF is obtained by adding, at each time step, the transition score between consecutive tags and the emission score of the current tag.

The score of a tag sequence $𝑦=(𝑦_1, \ldots, y_T)$ given emissions $E$ is:

$$Score(x,y) = start(y_1) + \sum_{t=1}^T{E_{t,y_t}} + \sum_{t=2}^T{A_{y_{t-1}, y_t}} + end(y_T)$$

where $E_{t,k}$ is the emission score for tag $k$ at position $t$
and $A_{i,j}$ is the transition score from tag $i$ to tag $j$.

**Exercise 7.** complete the implementation of the path score calculation

In [29]:
import torch
import torch.nn as nn

class CRF(nn.Module):
    def __init__(self, num_tags, batch_first=True):
        super().__init__()
        self.num_tags = num_tags
        self.batch_first = batch_first

        # transition scores
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def forward(self, emissions, tags, mask=None, reduction='mean'):
        if self.batch_first:
            emissions = emissions.transpose(0, 1)
            tags = tags.transpose(0, 1)
            if mask is not None:
                mask = mask.transpose(0, 1)

        if mask is None:
            mask = torch.ones_like(tags, dtype=torch.bool)

        log_den = self._compute_log_partition(emissions, mask)
        log_num = self._compute_score(emissions, tags, mask)
        llh = log_num - log_den

        if reduction == 'mean':
            return llh.mean()
        elif reduction == 'sum':
            return llh.sum()
        else:
            return llh

    def decode(self, emissions, mask=None):
        if self.batch_first:
            emissions = emissions.transpose(0, 1)
            if mask is not None:
                mask = mask.transpose(0, 1)

        if mask is None:
            mask = torch.ones(
                emissions.shape[:2],
                dtype=torch.bool,
                device=emissions.device
            )

        return self._viterbi_decode(emissions, mask)

    def _compute_score(self, emissions, tags, mask):
        seq_len, batch_size = tags.shape

        score = self.start_transitions[tags[0]]
        score += emissions[0, torch.arange(batch_size), tags[0]]

        for i in range(1, seq_len):
            # 1. Extraction vectorisée des scores de transition pour tout le batch
            transition_score = self.transitions[tags[i-1], tags[i]]
            
            # 2. Extraction vectorisée des scores d'émission pour le tag cible
            emission_score = emissions[i, torch.arange(batch_size), tags[i]]
            
            # 3. Calcul du score local pour l'étape i
            step_score = transition_score + emission_score
            
            # 4. Ajout au score global, conditionné par le masque (ignore le padding)
            score += step_score * mask[i]
            

        last_tag_indices = mask.long().sum(0) - 1
        last_tags = tags.gather(0, last_tag_indices.unsqueeze(0)).squeeze(0)

        score += self.end_transitions[last_tags]
        return score

    def _compute_log_partition(self, emissions, mask):
        seq_len = emissions.size(0)

        score = self.start_transitions + emissions[0]

        for i in range(1, seq_len):
            broadcast_score = score.unsqueeze(2)
            broadcast_emission = emissions[i].unsqueeze(1)
            next_score = broadcast_score + self.transitions + broadcast_emission

            next_score = torch.logsumexp(next_score, dim=1)
            score = torch.where(mask[i].unsqueeze(1), next_score, score)

        score += self.end_transitions
        return torch.logsumexp(score, dim=1)

    def _viterbi_decode(self, emissions, mask):
        seq_len, batch_size, num_tags = emissions.size()

        score = self.start_transitions + emissions[0]
        history = []

        for i in range(1, seq_len):
            broadcast_score = score.unsqueeze(2)
            next_score = broadcast_score + self.transitions
            best_score, best_tag = next_score.max(1)

            score = best_score + emissions[i]
            history.append(best_tag)

            score = torch.where(mask[i].unsqueeze(1), score, broadcast_score.squeeze(2))

        score += self.end_transitions
        best_last_score, best_last_tag = score.max(1)

        best_paths = []
        for b in range(batch_size):
            seq_end = mask[:, b].sum() - 1
            tag = best_last_tag[b].item()
            path = [tag]

            for hist in reversed(history[:seq_end]):
                tag = hist[b][tag].item()
                path.append(tag)

            best_paths.append(path[::-1])

        return best_paths


We now load the dataset in the following cell

In [30]:
import pandas as pd
import torch
import torch.nn as nn

# Load  CSV
df = pd.read_csv("ner_dataset.csv", encoding="latin1")

# Forward-fill the 'Sentence #' column so every row has a sentence ID
df = df.ffill()

# Group by Sentence # to create sequences
agg_func = lambda s: [(w, t) for w, t in zip(s["Word"].values.tolist(), s["Tag"].values.tolist())]
grouped = df.groupby("Sentence #").apply(agg_func)
sentences = [s for s in grouped]

# Build vocab and tag mappings
words = list(set(df["Word"].values))
tags = list(set(df["Tag"].values))

word2idx = {"PAD": 0, "UNK": 1}

for w in df["Word"].unique():
    if w not in word2idx:
        word2idx[w] = len(word2idx)

tag2idx = {t: i for i, t in enumerate(tags)}
idx2tag = {i: t for t, i in tag2idx.items()}

C:\Users\Alexander\AppData\Local\Temp\ipykernel_29144\2482884864.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = df.groupby("Sentence #").apply(agg_func)


**Exercise 8.** Implement the layers `lstm` (LSTM double layer, bidirectional, batch_first=True), `hidden2tag` (linear layer from hidden_dim to number of tags) and `crf` (batch_first=True)

In [31]:
class NER_Model(nn.Module):
    def __init__(self, vocab_size, num_tags, emb_dim=128, hidden_dim=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        
        # LSTM : 2 couches, bidirectionnel, batch en première dimension
        self.lstm = nn.LSTM(
            input_size=emb_dim, 
            hidden_size=hidden_dim // 2, 
            num_layers=2, 
            bidirectional=True, 
            batch_first=True
        )
        
        # Projection linéaire : de l'espace latent (caché) vers l'espace des tags
        self.hidden2tag = nn.Linear(
            in_features=hidden_dim, 
            out_features=num_tags
        )
        
        # Couche CRF (définie à l'exercice 7)
        self.crf = CRF(
            num_tags=num_tags, 
            batch_first=True
        )

    def forward(self, x, y=None, mask=None):
        emissions = self.hidden2tag(self.lstm(self.embedding(x))[0])
        if y is not None:
            return -self.crf(emissions, tags=y, mask=mask, reduction='mean')
        else:
            return self.crf.decode(emissions, mask=mask)

here we make a training/test split, load the model, train it and test on some sentences from the test data (note: if you use a large training set, this could take long)

In [32]:
import random

random.shuffle(sentences)

#split_idx = int(0.66 * len(sentences))
train_sentences = sentences[:1000]
test_sentences  = sentences[1000:]

print(f"Train: {len(train_sentences)}  Test: {len(test_sentences)}")


def prepare_sentence(sent):
    x = torch.tensor([word2idx.get(w[0], 1) for w in sent]).unsqueeze(0)
    y = torch.tensor([tag2idx[w[1]] for w in sent]).unsqueeze(0)
    mask = x.ne(0)
    return x, y, mask


model = NER_Model(len(word2idx), len(tags))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 2

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for sent in train_sentences:
        x, y, mask = prepare_sentence(sent)

        optimizer.zero_grad()
        loss = model(x, y, mask=mask)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | loss = {total_loss/len(train_sentences):.4f}")


model.eval()

with torch.no_grad():
    for sent in test_sentences[:3]:  # show a few examples
        x, y, mask = prepare_sentence(sent)

        best_path = model(x, mask=mask)[0]
        predicted_tags = [idx2tag[t] for t in best_path]

        print()
        print("Words :", [w[0] for w in sent])
        print("Gold  :", [w[1] for w in sent])
        print("Preds :", predicted_tags)

Train: 1000  Test: 46959
Epoch 1 | loss = 8.3480
Epoch 2 | loss = 3.7190

Words : ['The', 'bodies', 'of', 'two', 'of', 'the', 'other', 'hostages', 'were', 'handed', 'over', 'to', 'British', 'authorities', 'last', 'month', '.']
Gold  : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O']
Preds : ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O']

Words : ['Speaking', 'on', 'CNN', 'Sunday', ',', 'Congressman', 'Christopher', 'Shays', 'cited', 'continual', 'acts', 'of', 'Mr.', 'Delay', "'s", 'that', 'in', 'his', 'words', ',', '"', 'border', 'and', 'sometimes', 'go', 'beyond', '"', 'what', 'is', 'ethical', '.']
Gold  : ['O', 'O', 'B-org', 'B-tim', 'O', 'B-per', 'I-per', 'I-per', 'O', 'O', 'O', 'O', 'B-per', 'I-per', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Preds : ['O', 'O', 'B-geo', 'B-tim', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-per', 'I-per', 'O', 'O', 'O', 'O', 'O